In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#aqui foi retirado alguns outliers que não foram retirados na hora de montar a base hac
base_hac = pd.read_csv("base_hac.csv", index_col=0)

outliers_pvpa = ["SJAU11.SA", "PRSN11.SA"]
base_hac = base_hac.drop(index=outliers_pvpa)

print(f"Base após remoção: {len(base_hac)} FIIs")
print(f"P/VPA após remoção:")
print(base_hac["P_VPA"].describe().round(4))

base_hac.to_csv("base_hac.csv")
print("base_hac.csv atualizado.")

Base após remoção: 213 FIIs
P/VPA após remoção:
count    213.0000
mean       0.7893
std        0.2851
min        0.0498
25%        0.6690
50%        0.8154
75%        0.9253
max        2.7633
Name: P_VPA, dtype: float64
base_hac.csv atualizado.


In [ ]:
#Aqui são testados diferentes tipos de k, por meio de silhouette, Davies-Bouldin e WCV, e foi deduzido que o k de 4 era o ideal, porém a estrtura do mercado de fiis criou uma anomalia interessante,exite um grupo muito grande e homogeneo que se agrupou no cluster 1, e os outros clusters se mostram fiis em situações de distressed, alavancagem alta sensibilidade alta e por ai vai, ou seja, fiis em situações muito especificas.

# Carrega e padroniza a base
base      = pd.read_csv("base_hac.csv", index_col=0)
scaler    = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(base), index=base.index, columns=base.columns)
print(f"Base: {base.shape[0]} FIIs | {base.shape[1]} features | média={df_scaled.mean().mean():.4f} | std={df_scaled.std().mean():.4f}")

# Linkage de Ward sobre distância euclidiana
Z = linkage(pdist(df_scaled, metric="euclidean"), method="ward")
print(f"Linkage calculado: {len(Z)} fusões | última altura: {Z[-1, 2]:.4f}")

# Dendrograma
fig, ax = plt.subplots(figsize=(20, 8))
dendrogram(Z, labels=base.index.str.replace(".SA", ""), ax=ax,
           leaf_rotation=90, leaf_font_size=6, color_threshold=0)
ax.set_title("Dendrograma HAC — Ward Linkage")
ax.set_xlabel("Ticker")
ax.set_ylabel("Distância de Ward")
plt.tight_layout()
plt.savefig("dendrograma.png", dpi=150)
plt.close()

# Métricas para k de 2 a 12
print(f"\n{'k':<5} {'WCV':>10} {'Silhouette':>12} {'Davies-Bouldin':>16}")
resultados = []
for k in range(2, 13):
    labels = fcluster(Z, k, criterion="maxclust")
    wcv    = sum(np.var(df_scaled.values[labels == c], axis=0).sum() for c in range(1, k+1))
    sil    = silhouette_score(df_scaled, labels)
    db     = davies_bouldin_score(df_scaled, labels)
    resultados.append({"k": k, "wcv": wcv, "silhouette": sil, "davies_bouldin": db})
    print(f"{k:<5} {wcv:>10.2f} {sil:>12.4f} {db:>16.4f}")

df_metrics   = pd.DataFrame(resultados)
k_silhouette = df_metrics.loc[df_metrics["silhouette"].idxmax(), "k"]
k_db         = df_metrics.loc[df_metrics["davies_bouldin"].idxmin(), "k"]
print(f"\nSugestão: Silhouette → k={k_silhouette} | Davies-Bouldin → k={k_db}")

# Gráfico dos critérios
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, cor, titulo in zip(axes,
    ["wcv", "silhouette", "davies_bouldin"],
    ["blue", "green", "red"],
    ["Within-Cluster Variance (menor=melhor)",
     "Silhouette Score (maior=melhor)",
     "Davies-Bouldin Score (menor=melhor)"]):
    ax.plot(df_metrics["k"], df_metrics[col], marker="o", color=cor)
    ax.set_title(titulo)
    ax.set_xlabel("k")
    ax.set_xticks(df_metrics["k"])
plt.tight_layout()
plt.savefig("criterios_corte.png", dpi=150)
plt.close()

# Corte e atribuição de clusters
K_ESCOLHIDO   = 4
base["cluster"] = fcluster(Z, K_ESCOLHIDO, criterion="maxclust")
print(f"\nCorte k={K_ESCOLHIDO}:")
print(base["cluster"].value_counts().sort_index().to_string())

# Perfil médio por cluster
features = [c for c in base.columns if c != "cluster"]
perfil   = base.groupby("cluster")[features].mean().round(4)
print(f"\nPerfil médio por cluster:")
print(perfil.T.to_string())

# Heatmap do perfil
perfil_norm = (perfil - perfil.mean()) / perfil.std()
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(perfil_norm.T, annot=perfil.T.round(3), fmt=".3f",
            cmap="RdYlGn", center=0, ax=ax, linewidths=0.4)
ax.set_title(f"Perfil Médio por Cluster (k={K_ESCOLHIDO})")
plt.tight_layout()
plt.savefig("perfil_clusters.png", dpi=150)
plt.close()

# Exportação
base.to_csv("clusters_fiis.csv")
df_metrics.to_csv("metricas_corte.csv", index=False)
print(f"\nSalvos: clusters_fiis.csv | metricas_corte.csv | dendrograma.png | criterios_corte.png | perfil_clusters.png")

Base: 213 FIIs | 8 features | média=0.0000 | std=1.0024
Linkage calculado: 212 fusões | última altura: 19.2642

k            WCV   Silhouette   Davies-Bouldin
2          27.69       0.4253           2.0988
3          37.20       0.3987           1.5800
4          40.57       0.3906           1.2287
5          50.05       0.3256           1.3749
6          56.56       0.3255           1.3252
7          59.08       0.1230           1.5250
8          62.74       0.1348           1.4087
9          63.56       0.1421           1.3343
10         67.60       0.1421           1.2872
11         68.24       0.1289           1.2269
12         65.71       0.1347           1.1513

Sugestão: Silhouette → k=2 | Davies-Bouldin → k=12

Corte k=4:
cluster
1    190
2      3
3      7
4     13

Perfil médio por cluster:
cluster                        1       2       3       4
DY_Mes                    0.0070  0.0090  0.0077  0.0072
LTV                       0.0473  0.0935  0.5341  0.0692
Percentual_Vacanci

In [ ]:
#Aqui foi aplicado o uma clusterização do cluster 1, para identificar se existe alguma divisãon um pouco mais profunda dentro do grupo mainstream. Novamente utilizando os mesmos metodos que antes foi escolhido o k=7 para a clusterização. Assim, deu para ver um nivel de separação maior, porém ainda com um grupo muito homogeneo de 125 fiis muito parecidos. Mostrando ao contrario do imaginado, o mercado de fiis como um todo, responde igual a choques e a diversificação via setores, na prática não é uma diversificação real.
# Isola o cluster mainstream e padroniza internamente
idx_c1    = clusters[clusters["cluster"] == 1].index
base_c1   = base.loc[idx_c1].copy()
scaler    = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(base_c1), index=base_c1.index, columns=base_c1.columns)
print(f"Subclustering: {len(base_c1)} FIIs")

# Linkage de Ward no subgrupo
Z_sub = linkage(pdist(df_scaled, metric="euclidean"), method="ward")

# Métricas para k de 2 a 8
print(f"\n{'k':<5} {'WCV':>10} {'Silhouette':>12} {'Davies-Bouldin':>16}")
resultados = []
for k in range(2, 9):
    labels = fcluster(Z_sub, k, criterion="maxclust")
    wcv    = sum(np.var(df_scaled.values[labels == c], axis=0).sum() for c in range(1, k+1))
    sil    = silhouette_score(df_scaled, labels)
    db     = davies_bouldin_score(df_scaled, labels)
    resultados.append({"k": k, "wcv": wcv, "silhouette": sil, "davies_bouldin": db})
    print(f"{k:<5} {wcv:>10.2f} {sil:>12.4f} {db:>16.4f}")

df_sub_metrics = pd.DataFrame(resultados)
k_sil = df_sub_metrics.loc[df_sub_metrics["silhouette"].idxmax(), "k"]
k_db  = df_sub_metrics.loc[df_sub_metrics["davies_bouldin"].idxmin(), "k"]
print(f"\nSugestão: Silhouette → k={k_sil} | Davies-Bouldin → k={k_db}")

# Aplica k=7 e atribui subclusters
K_SUB          = 7
sub_labels     = fcluster(Z_sub, K_SUB, criterion="maxclust")
base_c1["subcluster"] = sub_labels

print(f"\nDistribuição k={K_SUB}:")
for c in range(1, K_SUB + 1):
    print(f"  Subcluster {c}: {(sub_labels == c).sum()} FIIs")

# Perfil médio por subcluster
features   = [c for c in base_c1.columns if c != "subcluster"]
perfil_sub = base_c1.groupby("subcluster")[features].mean().round(4)
print(f"\nPerfil médio por subcluster:")
print(perfil_sub.T.to_string())

# Heatmap
perfil_norm = (perfil_sub - perfil_sub.mean()) / perfil_sub.std()
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(perfil_norm.T, annot=perfil_sub.T.round(3), fmt=".3f",
            cmap="RdYlGn", center=0, ax=ax, linewidths=0.4)
ax.set_title(f"Perfil Subclusters do Cluster 1 (k={K_SUB})")
plt.tight_layout()
plt.savefig("perfil_subclusters.png", dpi=150)
plt.close()
print("Salvo: perfil_subclusters.png")

Subclustering: 190 FIIs

k            WCV   Silhouette   Davies-Bouldin
2          18.32       0.3354           1.6819
3          24.65       0.2236           1.7603
4          25.26       0.2438           1.4494
5          35.53       0.2572           1.4924
6          42.36       0.2153           1.3771
7          47.12       0.2266           1.2133
8          50.51       0.2166           1.2773

Sugestão: Silhouette → k=2 | Davies-Bouldin → k=7

Distribuição k=7:
  Subcluster 1: 12 FIIs
  Subcluster 2: 8 FIIs
  Subcluster 3: 21 FIIs
  Subcluster 4: 7 FIIs
  Subcluster 5: 125 FIIs
  Subcluster 6: 2 FIIs
  Subcluster 7: 15 FIIs

Perfil médio por subcluster:
subcluster                     1       2       3       4       5       6       7
DY_Mes                    0.0051  0.0055  0.0073  0.0266  0.0066  0.0000  0.0043
LTV                       0.0149  0.0632  0.1855  0.1148  0.0255  0.0689  0.0188
Percentual_Vacancia       0.1874  0.0492  0.0357  0.0377  0.0037  0.0000  0.0108
Percentua

In [ ]:
#aqui atrubui os arquétipos possiveis para os clusters, com base nas respostas as features, levando em consideração a interpretaçãoe econômica.
# Isola cluster 1 e roda subclustering com k=7
idx_c1    = clusters_main[clusters_main["cluster"] == 1].index
base_c1   = base.loc[idx_c1].copy()
df_scaled = pd.DataFrame(
    StandardScaler().fit_transform(base_c1),
    index=base_c1.index, columns=base_c1.columns
)
base_c1["subcluster"] = fcluster(
    linkage(pdist(df_scaled, metric="euclidean"), method="ward"),
    7, criterion="maxclust"
)

# Nomes econômicos dos arquétipos
nomes_sub = {
    1: "Tijolo com Vacância",
    2: "Papel Crédito Moderado",
    3: "Alavancado Moderado",
    4: "Income Premium",
    5: "Core Conservador",
    6: "CRI IPCA+",
    7: "Duration Longo",
}
nomes_extremos = {
    2: "Crédito Distressed",
    3: "Alavancagem Extrema",
    4: "Imóvel Degradado",
}

# Atribui arquétipos
clusters_main["arquetipo"] = None
for c, nome in nomes_extremos.items():
    clusters_main.loc[clusters_main["cluster"] == c, "arquetipo"] = nome
for ticker in base_c1.index:
    clusters_main.loc[ticker, "arquetipo"] = nomes_sub[int(base_c1.loc[ticker, "subcluster"])]

# Junta com a base de features
base_final = base.join(clusters_main[["cluster", "arquetipo"]])

# Distribuição e perfil
print("Distribuição por arquétipo:")
print(base_final["arquetipo"].value_counts().to_string())
print(f"\nTotal: {len(base_final)} FIIs | {base_final['arquetipo'].nunique()} arquétipos")

perfil      = base_final.groupby("arquetipo")[list(base.columns)].mean().round(4)
perfil_norm = (perfil - perfil.mean()) / perfil.std()
print(f"\nPerfil médio por arquétipo:")
print(perfil.T.to_string())

# Heatmap final
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(perfil_norm.T, annot=perfil.T.round(3), fmt=".3f",
            cmap="RdYlGn", center=0, ax=ax, linewidths=0.4)
ax.set_title("Perfil Médio por Arquétipo — 10 Grupos | 213 FIIs")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig("perfil_arquetipos_final.png", dpi=150)
plt.close()

# Exporta
base_final.to_csv("clusters_fiis_final.csv")
print("\nSalvos: clusters_fiis_final.csv | perfil_arquetipos_final.png")

Distribuição por arquétipo:
arquetipo
Core Conservador          125
Alavancado Moderado        21
Duration Longo             15
Imóvel Degradado           13
Tijolo com Vacância        12
Papel Crédito Moderado      8
Alavancagem Extrema         7
Income Premium              7
Crédito Distressed          3
CRI IPCA+                   2

Total: 213 FIIs | 10 arquétipos

Perfil médio por arquétipo:
arquetipo                 Alavancado Moderado  Alavancagem Extrema  CRI IPCA+  Core Conservador  Crédito Distressed  Duration Longo  Imóvel Degradado  Income Premium  Papel Crédito Moderado  Tijolo com Vacância
DY_Mes                                 0.0073               0.0077     0.0000            0.0066              0.0090          0.0043            0.0072          0.0266                  0.0055               0.0051
LTV                                    0.1855               0.5341     0.0689            0.0255              0.0935          0.0188            0.0692          0.1148             

In [ ]:
#aqui eu quis verificar o dp médio das features dentro dos clusters, para ter certeza que os agrupamentos fazem sentido. Cluster mais externos devem ter dp maiores e clusters mais internos como os mainstream, devem ter os menores.
base  = pd.read_csv("base_hac.csv", index_col=0)
final = pd.read_csv("clusters_fiis_final.csv", index_col=0)

base_final = base.join(final[["arquetipo"]])

print("Desvio padrão interno por arquétipo (escala original):")


resultado = []
for arq in base_final["arquetipo"].unique():
    grupo = base_final[base_final["arquetipo"] == arq].drop(columns="arquetipo")
    n     = len(grupo)
    std_medio = grupo.std().mean()
    resultado.append({"arquetipo": arq, "n": n, "std_medio": std_medio})

df_res = (pd.DataFrame(resultado)
            .sort_values("std_medio")
            .reset_index(drop=True))

print(df_res.to_string(index=False))

Desvio padrão interno por arquétipo (escala original):
             arquetipo   n  std_medio
      Core Conservador 125   0.034366
   Tijolo com Vacância  12   0.039259
   Alavancado Moderado  21   0.041786
Papel Crédito Moderado   8   0.051388
        Duration Longo  15   0.077134
   Alavancagem Extrema   7   0.083102
             CRI IPCA+   2   0.089549
        Income Premium   7   0.091918
      Imóvel Degradado  13   0.118950
    Crédito Distressed   3   0.188738
